# D1.2 · Context that makes triage work

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *AI for Security*

Builds on **[D1.1 · From alert queue to loop operator](https://spbreed.github.io/cyber-commons/lessons/D1.1.html)**.

| | |
|---|---|
| Tools used | Wazuh, GLM-4.6, Llama 3.3, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** A/B a generic prompt vs a context-loaded one on the same alert set.

**Why a security engineer needs it.** Generic triage agents underperform your worst analyst. The control it builds is: feed the baseline, known FPs, crown-jewel map and prior decisions.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Most bad triage is not a bad model. It is an agent asked to decide without the identity, asset and history context a human analyst would have pulled without noticing they pulled it.

> **At CyberTravels.** An alert saying `cybertravels-svc listed all customer records` is untriageable without knowing whether that is its job. Most bad triage at CyberTravels is missing context, not a weak model.

## 2 · The framework

```
   the alert                what a human would have pulled without thinking
   +----------------+       +-----------------------------------+
   | user: dana     |  -->  | is dana on call?                  |
   | host: build-07 |       | is build-07 a build agent?        |
   | 03:14          |       | has this fired for dana before?   |
   +----------------+       +-----------------------------------+

   most bad triage is missing context, not a weak model
```

An alert about a human is triageable with three facts: who, what, when. An alert
about an agent needs three more, and without them every analyst has to guess.

- **The acting identity** and the principal it acted for (A2.1).
- **The scopes it held** at the time. This is the decisive field: reading
  `.env` is alarming for an agent scoped `repo:read` and routine for a
  secrets-rotation agent.
- **The delegation chain**, so the analyst can see who caused the task.

Without scope in the alert, the analyst's only options are to escalate
everything or to develop a habit of closing agent alerts. Both happen, and the
second one happens quietly.

## 3 · Triage as a skill — and the sample that keeps it honest

Context turns a guess into a verdict. Automating the verdict without automating the audit of it is how a closing rule quietly starts closing real incidents.

The skill therefore requires a sampling rule over anything auto-closed, and requires its seed to come from something **stable**. Sampling seeded from `hash()` picks a different subset on every run, so you can never tell whether a change in findings came from the rule or from the dice.

### The skill — [`skills/secops/detection-triage/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/secops/detection-triage/SKILL.md)

```yaml
name: detection-triage
description: >-
  Triage security alerts with the context needed to reach a defensible verdict,
  and sample what is auto-closed so the closing rule stays honest. Use when
  working an alert queue, deciding whether an alert is a true positive, tuning
  a noisy detection, or designing automated alert handling.
allowed-tools: Read, Grep, Bash
```

# Alert triage with context

An analyst reading an alert in isolation is guessing. The verdict comes from
the alert **plus** the context that makes it normal or abnormal, and most
triage automation fails because it automated the guess instead of the context.

## When to use this

Working an alert queue, building an auto-close rule, or reviewing why a
detection produces verdicts nobody trusts.

## Procedure

**1 — Gather the context before judging.** For every alert, assemble:

- **asset** — what it is, who owns it, how exposed it is
- **identity** — human or workload, and its normal behaviour
- **history** — has this fired before on this asset, and how was it resolved
- **change** — was there a deploy, a migration, a new agent, in the window
- **peers** — did the same thing fire elsewhere at the same time

A verdict issued without `history` will re-litigate a decision the team already
made, which is the most common way triage automation loses trust.

**2 — Reach a verdict, with the reason.** One of `true_positive`,
`false_positive`, `benign_true_positive` (it really happened and it is fine),
or `needs_human`. Record which context field decided it. "Benign true positive"
is a distinct category and collapsing it into false-positive corrupts every
tuning decision made from the data afterwards.

**3 — Attach confidence, and let it gate automation.** Only high-confidence
verdicts may auto-close. Everything else queues.

**4 — Sample the auto-closed.** Automation that closes alerts must be audited
by re-opening a fraction of them for human review. This is the control that
catches a closing rule that has quietly started closing real incidents.

Seed the sampler from something **stable** — a checksum of the alert id, never
`hash()`, which Python randomises per process. A sampling rule that picks a
different subset every run cannot be audited, because you cannot tell whether a
change in findings came from the rule or from the dice.

**5 — Feed tuning from verdicts, not volume.** A detection is noisy if its
false-positive rate is high, not if it fires often. Rank tuning candidates by
`false_positive_rate × volume`, with a full tiebreak so the list is stable
between runs.

## Example

**Input** — the fixture committed at the top of [`scripts/detection_triage.py`](scripts/detection_triage.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
BARE ALERT (what most SOCs receive):
   patch-agent read /vault/.env
   rotator-agent read /vault/.env
   → identical. An analyst cannot tell these apart.

ENRICHED ALERT:
   actor        patch-agent
   on behalf of dana@corp
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "triaged": [
    {"alert_id": "str", "verdict": "true_positive|false_positive|benign_true_positive|needs_human",
     "deciding_context": "asset|identity|history|change|peers",
     "reason": "str", "confidence": 0.0,
     "auto_closed": false, "sampled_for_review": false}
  ],
  "sampling": {"rate": 0.0, "seed_source": "str", "reviewed": 0, "disagreements": 0},
  "tuning": [{"rule": "str", "fp_rate": 0.0, "volume": 0, "priority": 0.0}]
}
```

`disagreements` is the number that matters: it is the measured error rate of
the automation, and it belongs in every report about it.

## Failure modes

- **Auto-closing without sampling.** The rule then has no error bar and no way
  to acquire one.
- **Seeding the sampler from `hash()`.** Non-reproducible sampling is not a
  control.
- **Folding benign-true-positive into false-positive.** It teaches the tuner to
  suppress a working detection.
- **Removing `needs_human`.** Forced verdicts under uncertainty are how a queue
  becomes an incident.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/secops/detection-triage/scripts/detection_triage.py
SCRIPT = "skills/secops/detection-triage/scripts/detection_triage.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The bare alert is identical for both agents. Enriched, the secrets-rotation agent is within remit and the patch agent is not. Context-free triage escalates both — generating a nightly false positive — while scope-aware triage matches ground truth on both.

## Your turn

Check which of the six fields your agent telemetry carries today. Scopes-held is the one almost nobody logs, and it is the one that decides the alert.

---

**Next → [D1.3 · Agent-assisted detection engineering](https://spbreed.github.io/cyber-commons/lessons/D1.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*